# TT-05 — SVM: Phân loại khối u lành tính / ác tính
### Breast Cancer Wisconsin (Diagnostic) — theo README.md


## Bước 0 — Cài thư viện (chỉ cần chạy 1 lần)
> Dùng `%pip` (không phải `!pip`) để cài đúng vào môi trường của kernel đang chọn.
> Sau khi cài xong, **Restart Kernel** rồi mới chạy tiếp các ô bên dưới.

In [ ]:
%pip install scikit-learn pandas matplotlib seaborn joblib

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, recall_score, precision_score,
    ConfusionMatrixDisplay, make_scorer
)

# Notebook nằm trong notebooks/ -> lùi ra 1 cấp để về thư mục gốc project
os.makedirs("../reports", exist_ok=True)
os.makedirs("../models", exist_ok=True)

print("Thư mục làm việc hiện tại:", os.getcwd())


## Bước 1 — Nạp dữ liệu, xác nhận quy ước nhãn (0 = ác tính!)
⚠️ Trong sklearn: **0 = malignant (ác tính)**, **1 = benign (lành tính)** — dễ nhầm dấu!

In [ ]:
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target

print("Kích thước X:", X.shape)
print("Tên nhãn (target_names):", data.target_names)
print("\nPhân bố nhãn:")
print(y.value_counts())


### Kiểm tra thêm thông tin dataset

In [ ]:
print("Kiểu dữ liệu từng cột:")
print(X.dtypes.value_counts())

print("\nSố giá trị thiếu (NaN):", X.isnull().sum().sum())

print("\nThống kê mô tả (mean, std, min, max):")
X.describe().T[["mean", "std", "min", "max"]]


## Bước 2 — EDA: heatmap tương quan 30 biến
30 biến = 10 chỉ số × 3 thống kê (mean / standard error / worst) → nhiều cột tương quan rất mạnh (`radius_mean` ↔ `perimeter_mean` ↔ `area_mean`).

In [ ]:
plt.figure(figsize=(14, 12))
sns.heatmap(X.corr(), cmap="coolwarm", center=0)
plt.title("Tương quan giữa 30 đặc trưng")
plt.tight_layout()
plt.savefig("../reports/correlation_heatmap.png")
plt.show()


## Bước 3 — Chia train/test (stratify theo y)
`stratify=y` đảm bảo tỉ lệ ác tính/lành tính ở tập test giống tập gốc.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, "| Test:", X_test.shape)


## Bước 4 — ⚠️ SVM KHÔNG scale (baseline)
Chạy để **chứng minh bằng số liệu** vì sao SVM cần chuẩn hoá — không chỉ nói suông.

In [ ]:
svm_noscale = SVC(random_state=42)
svm_noscale.fit(X_train, y_train)
pred_noscale = svm_noscale.predict(X_test)

print(classification_report(y_test, pred_noscale, target_names=data.target_names))

recall_noscale = recall_score(y_test, pred_noscale, pos_label=0)
print(f"Recall lớp ÁC TÍNH (không scale): {recall_noscale:.3f}")


## Bước 5 — SVM CÓ scale (Pipeline) — so sánh với Bước 4
`Pipeline` đảm bảo `StandardScaler` chỉ học (fit) từ `X_train`, tránh rò rỉ dữ liệu (data leakage) sang `X_test`.

`class_weight='balanced'` giúp tăng phạt khi đoán sai lớp thiểu số (ác tính), kéo recall lên.

In [ ]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("svm", SVC(class_weight="balanced", random_state=42)),
])
pipe.fit(X_train, y_train)
pred_scaled = pipe.predict(X_test)

print(classification_report(y_test, pred_scaled, target_names=data.target_names))

recall_scaled = recall_score(y_test, pred_scaled, pos_label=0)
print(f"Recall lớp ÁC TÍNH (có scale): {recall_scaled:.3f}")


### Bảng + biểu đồ so sánh CÓ vs KHÔNG scale

In [ ]:
compare_scale = pd.DataFrame({
    "Không scale": [recall_noscale],
    "Có scale": [recall_scaled],
}, index=["Recall (ác tính)"])
print(compare_scale)

plt.figure(figsize=(5, 4))
compare_scale.T.plot(kind="bar", legend=False)
plt.ylabel("Recall lớp ác tính")
plt.title("SVM: Có scale vs Không scale")
plt.tight_layout()
plt.savefig("../reports/scale_vs_noscale.png")
plt.show()


## Bước 6 — So sánh 3 kernel (cùng C=1)
- `linear`: giả định 2 lớp tách được bằng mặt phẳng thẳng
- `rbf`: ranh giới cong, linh hoạt — thường là lựa chọn mặc định tốt
- `poly`: quan hệ đa thức, dễ overfit hơn nếu `degree` cao

In [ ]:
kernel_results = {}
for kernel in ["linear", "rbf", "poly"]:
    p = Pipeline([
        ("scale", StandardScaler()),
        ("svm", SVC(kernel=kernel, C=1, class_weight="balanced", random_state=42)),
    ])
    p.fit(X_train, y_train)
    pred = p.predict(X_test)
    kernel_results[kernel] = recall_score(y_test, pred, pos_label=0)

kernel_df = pd.Series(kernel_results, name="Recall (ác tính)")
print(kernel_df)

plt.figure(figsize=(5, 4))
kernel_df.plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A868"])
plt.ylabel("Recall lớp ác tính")
plt.title("So sánh kernel (C=1)")
plt.tight_layout()
plt.savefig("../reports/kernel_comparison.png")
plt.show()


## Bước 7 — GridSearchCV dò C, gamma, kernel
⚠️ `scoring='recall'` mặc định tính recall của lớp `1`. Ta cần recall của lớp **0 (ác tính)** nên phải dùng `make_scorer(recall_score, pos_label=0)` — đây chính là cạm bẫy "nhầm quy ước nhãn 0/1" README cảnh báo.

In [ ]:
recall_malignant_scorer = make_scorer(recall_score, pos_label=0)

grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.001, 0.01, 0.1],
    "svm__kernel": ["linear", "rbf", "poly"],
}

gs = GridSearchCV(pipe, grid, cv=5, scoring=recall_malignant_scorer, n_jobs=-1)
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print("Best CV recall (ác tính):", round(gs.best_score_, 4))


## Bước 8 — Heatmap điểm CV theo (C, gamma) — kernel RBF
Giúp nhìn thấy **vùng** tham số tốt, không chỉ 1 điểm may mắn.

In [ ]:
cv_results = pd.DataFrame(gs.cv_results_)
rbf_results = cv_results[cv_results["param_svm__kernel"] == "rbf"]

pivot = rbf_results.pivot_table(
    index="param_svm__gamma", columns="param_svm__C", values="mean_test_score"
)

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="viridis")
plt.title("Recall CV (ác tính) theo C và gamma — kernel RBF")
plt.tight_layout()
plt.savefig("../reports/C_gamma_heatmap.png")
plt.show()


## Bước 9 — Đếm support vectors
% support vector cao (gần 100%) → model gần như "nhớ" hết dữ liệu → dấu hiệu overfit.

In [ ]:
best_svm = gs.best_estimator_.named_steps["svm"]
n_sv = best_svm.n_support_
pct_sv = n_sv.sum() / len(X_train) * 100

print(f"Support vectors mỗi lớp: {n_sv}")
print(f"Tổng support vectors: {n_sv.sum()} / {len(X_train)} ({pct_sv:.1f}%)")


## Bước 10 — Chọn ngưỡng để recall ác tính ≥ 0.98
Hạ ngưỡng quyết định để bắt được nhiều ca ác tính hơn (đánh đổi bằng precision thấp hơn).

⚠️ `probability=True` chạy chậm hơn (Platt scaling nội bộ) — chỉ bật khi cần bước này.

In [ ]:
best_params_clean = {k.replace("svm__", ""): v for k, v in gs.best_params_.items()}

pipe_proba = Pipeline([
    ("scale", StandardScaler()),
    ("svm", SVC(**best_params_clean, probability=True,
                class_weight="balanced", random_state=42)),
])
pipe_proba.fit(X_train, y_train)

# proba[:, 0] = xác suất thuộc lớp 0 (ác tính)
proba_malignant = pipe_proba.predict_proba(X_test)[:, 0]

threshold_results = []
chosen_threshold = None
for threshold in np.arange(0.05, 0.55, 0.05):
    pred_thresh = np.where(proba_malignant >= threshold, 0, 1)
    r = recall_score(y_test, pred_thresh, pos_label=0)
    p = precision_score(y_test, pred_thresh, pos_label=0)
    threshold_results.append((round(threshold, 2), r, p))
    if r >= 0.98 and chosen_threshold is None:
        chosen_threshold = threshold

threshold_df = pd.DataFrame(threshold_results, columns=["threshold", "recall", "precision"])
print(threshold_df)

if chosen_threshold is None:
    chosen_threshold = threshold_df["threshold"].min()
    print("\nCẢNH BÁO: không tìm được threshold đạt recall >= 0.98 trong khoảng quét,"
          " dùng threshold nhỏ nhất đã thử.")
else:
    r_chosen = threshold_df.loc[threshold_df.threshold == round(chosen_threshold, 2), "recall"].values[0]
    print(f"\n-> Chọn threshold = {chosen_threshold:.2f} (recall ác tính = {r_chosen:.3f})")


## Bước 11 — Ma trận nhầm lẫn (sau khi chỉnh ngưỡng)
Ghi rõ: **số ca ác tính bị bỏ sót = ô (thực tế ác tính, dự đoán lành tính)**. Lý tưởng = 0.

In [ ]:
pred_final = np.where(proba_malignant >= chosen_threshold, 0, 1)

cm_disp = ConfusionMatrixDisplay.from_predictions(
    y_test, pred_final, display_labels=data.target_names, cmap="Blues"
)
plt.title(f"Ma trận nhầm lẫn (threshold={chosen_threshold:.2f})")
plt.tight_layout()
plt.savefig("../reports/confusion_matrix.png")
plt.show()

cm = cm_disp.confusion_matrix
missed_malignant = cm[0, 1]  # thực tế ác tính (0), dự đoán lành tính (1)
print(f"Số ca ÁC TÍNH bị bỏ sót: {missed_malignant}")


## Bước 12 — So sánh với Logistic Regression (TT-04)

In [ ]:
logreg = Pipeline([
    ("scale", StandardScaler()),
    ("lr", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
logreg.fit(X_train, y_train)
pred_lr = logreg.predict(X_test)
recall_lr = recall_score(y_test, pred_lr, pos_label=0)

recall_svm_final = recall_score(y_test, pred_final, pos_label=0)

print(f"Logistic Regression - recall (ác tính): {recall_lr:.3f}")
print(f"SVM (đã chỉnh ngưỡng)  - recall (ác tính): {recall_svm_final:.3f}")


**Nhận xét (tự viết vào báo cáo):** với 569 dòng, 30 đặc trưng khá tách bạch tuyến tính, Logistic Regression thường cho kết quả gần tương đương SVM nhưng dễ giải thích hơn (có hệ số cho từng biến). So sánh số liệu thực tế chạy ra ở trên để đưa ra kết luận của riêng bạn.

## Lưu model cuối cùng

In [ ]:
joblib.dump(gs.best_estimator_, "../models/svm_pipeline.joblib")
print("Đã lưu models/svm_pipeline.joblib")
